Đơn luồng 

In [ ]:
import requests
import csv
import os
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

def load_urls(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def create_filename(url):
    return url.replace("http://", "").replace("https://", "").replace("/", "_") + ".html"

def fetch_html(url, retries=3):
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)

            if response.status_code == 200:
                return response.text
            else:
                print(f"[WARN] {url} -> status {response.status_code}")

        except requests.RequestException as e:
            print(f"[RETRY {attempt+1}] {url} -> {e}")
            time.sleep(1)

    return None

def save_html(content, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

def main():
    # urls = load_urls("urls.txt")
    urls = load_urls("urls_alive.txt")

    os.makedirs("html_pages", exist_ok=True)

    with open("defaced_data_clean.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["URL", "HTML_File_Name"])

        for url in urls:
            html = fetch_html(url)

            if html:
                filename = create_filename(url)
                path = os.path.join("html_pages", filename)

                save_html(html, path)
                writer.writerow([url, filename])

                print(f"[OK] {url}")
            else:
                print(f"[FAILED] {url}")

if __name__ == "__main__":
    main()

Đa luồng

In [ ]:
import requests
import csv
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

def load_urls(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def create_filename(url):
    return url.replace("http://", "").replace("https://", "").replace("/", "_") + ".html"

def fetch_html(url, retries=3):
    for attempt in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, timeout=5)

            if res.status_code == 200:
                return url, res.text
        except:
            pass

    return url, None

def save_result(result):
    url, html = result

    if not html:
        print(f"[FAILED] {url}")
        return None

    filename = create_filename(url)
    path = os.path.join("html_pages", filename)

    with open(path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"[OK] {url}")
    return (url, filename)

def main():
    urls = load_urls("urls_alive.txt")

    os.makedirs("html_pages", exist_ok=True)

    results = []

    # số thread (có thể chỉnh)
    num_threads = 10

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(fetch_html, url) for url in urls]

        for future in as_completed(futures):
            result = save_result(future.result())
            if result:
                results.append(result)

    # ghi CSV sau khi xong
    with open("defaced_data_clean.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["URL", "HTML_File_Name"])
        writer.writerows(results)

if __name__ == "__main__":
    main()

Theo dõi tiến độ 

In [2]:
import requests
import csv
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

lock = threading.Lock()
completed = 0

def load_urls(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def create_filename(url):
    return url.replace("http://", "").replace("https://", "").replace("/", "_") + ".html"

def fetch_html(url, retries=3):
    for _ in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, timeout=(3,5))
            if res.status_code == 200:
                return url, res.text
        except Exception:
            pass
    return url, None

def save_result(result, total):
    global completed

    url, html = result

    with lock:
        completed += 1
        percent = (completed / total) * 100

    if not html:
        print(f"[{completed}/{total} - {percent:.1f}%] FAILED: {url}")
        return None

    filename = create_filename(url)
    path = os.path.join("html_pages", filename)

    with open(path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"[{completed}/{total} - {percent:.1f}%] OK: {url}")
    return (url, filename)

def main():
    urls = load_urls("urls_alive.txt")
    total = len(urls)

    os.makedirs("html_pages", exist_ok=True)

    results = []

    num_threads = 10

    print(f"Total URLs: {total}")
    print(f"Running with {num_threads} threads...\n")

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(fetch_html, url) for url in urls]

        for future in as_completed(futures):
            result = save_result(future.result(), total)
            if result:
                results.append(result)

    with open("defaced_data_clean.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["URL", "HTML_File_Name"])
        writer.writerows(results)

    print("\nDone!")

if __name__ == "__main__":
    main()

Total URLs: 27
Running with 10 threads...

[1/27 - 3.7%] OK: http://www.kejari-sleman.go.id
[2/27 - 7.4%] OK: http://commandcenter.bimakota.go.id
[3/27 - 11.1%] OK: http://twoguysandabucket.com
[4/27 - 14.8%] OK: http://siggec.gov.ao
[5/27 - 18.5%] OK: http://revistas.face.ufmg.br
[6/27 - 22.2%] OK: http://rompetesgrow.com.ar
[7/27 - 25.9%] FAILED: http://alternativaradio.fm
[8/27 - 29.6%] OK: http://revistas.ufrj.br
[9/27 - 33.3%] FAILED: http://apps.telessaude.hc.ufmg.br
[10/27 - 37.0%] FAILED: http://faminvestment.ae
[11/27 - 40.7%] OK: http://behalinternational.com
[12/27 - 44.4%] OK: http://periodicos.ufal.br
[13/27 - 48.1%] OK: http://espb.ao
[14/27 - 51.9%] OK: http://thebloomingstoryindia.com
[15/27 - 55.6%] FAILED: http://network.hr
[16/27 - 59.3%] OK: http://fastbooks.info
[17/27 - 63.0%] OK: http://perfectlifestyle.info
[18/27 - 66.7%] OK: http://solusiherbalalami.com
[19/27 - 70.4%] OK: http://elementsrealfood.com
[20/27 - 74.1%] OK: http://wps-grup.ro
[21/27 - 77.8%] OK: h

In [ ]:
import requests
import csv
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

lock = threading.Lock()
completed = 0
success_count = 0
fail_count = 0

def load_urls(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def create_filename(url):
    return url.replace("http://", "").replace("https://", "").replace("/", "_") + ".html"

def fetch_html(url, retries=3):
    for _ in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, timeout=(3,5))
            if res.status_code == 200:
                return url, res.text
        except Exception:
            pass
    return url, None

def save_result(result, total):
    global completed, success_count, fail_count

    url, html = result

    with lock:
        completed += 1

        if html:
            success_count += 1
        else:
            fail_count += 1

        percent = (completed / total) * 100

    if not html:
        print(f"[{completed}/{total} - {percent:.1f}%] FAILED: {url}")
        return None

    filename = create_filename(url)
    path = os.path.join("html_pages", filename)

    with open(path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"[{completed}/{total} - {percent:.1f}%] OK: {url}")
    return (url, filename)

def main():
    urls = load_urls("urls_alive.txt")
    total = len(urls)

    os.makedirs("html_pages", exist_ok=True)

    results = []
    num_threads = 10

    print(f"Total URLs: {total}")
    print(f"Running with {num_threads} threads...\n")

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(fetch_html, url) for url in urls]

        for future in as_completed(futures):
            result = save_result(future.result(), total)
            if result:
                results.append(result)

    with open("defaced_data_clean.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["URL", "HTML_File_Name"])
        writer.writerows(results)

    print("\n===== SUMMARY =====")
    print(f"Total: {total}")
    print(f"Success: {success_count}")
    print(f"Failed: {fail_count}")
    # print(f"Success rate: {(success_count/total)*100:.2f}%")

    print("\nDone!")

if __name__ == "__main__":
    main()

Total URLs: 27
Running with 10 threads...

[1/27 - 3.7%] OK: http://www.kejari-sleman.go.id
[2/27 - 7.4%] OK: http://commandcenter.bimakota.go.id
[3/27 - 11.1%] OK: http://behalinternational.com
[4/27 - 14.8%] OK: http://twoguysandabucket.com
[5/27 - 18.5%] FAILED: http://alternativaradio.fm
[6/27 - 22.2%] OK: http://siggec.gov.ao
[7/27 - 25.9%] OK: http://rompetesgrow.com.ar
[8/27 - 29.6%] FAILED: http://apps.telessaude.hc.ufmg.br
[9/27 - 33.3%] FAILED: http://faminvestment.ae
[10/27 - 37.0%] OK: http://revistas.face.ufmg.br
[11/27 - 40.7%] OK: http://fastbooks.info
[12/27 - 44.4%] OK: http://elementsrealfood.com
[13/27 - 48.1%] OK: http://perfectlifestyle.info
[14/27 - 51.9%] OK: http://espb.ao
[15/27 - 55.6%] OK: http://holyspain.com
[16/27 - 59.3%] OK: http://revistas.ufrj.br
[17/27 - 63.0%] FAILED: http://network.hr
[18/27 - 66.7%] OK: http://periodicos.ufal.br
[19/27 - 70.4%] OK: http://solusiherbalalami.com
[20/27 - 74.1%] OK: http://thebloomingstoryindia.com
[21/27 - 77.8%] OK: